# NIH ChestX-ray14 final patch — phenotype/view matching + deterministic paired rerun + corrected XAI

This notebook is the **minimal-compute final patch** for the executed
`nih-x-ray-improved-final.ipynb` experiment.

It is intentionally designed to use the **OUTPUT of the executed main run as a Kaggle input**. The
already-clean 6,000-image cohort and fixed primary test/validation/rigorous-training manifests are
reused. Old model predictions, old model checkpoints, and old figures are **not** reused.

The same raw NIH ChestX-ray14 image dataset must also be attached because the final paired models
and Grad-CAM analysis still require the radiographs.

## Why a patch is still useful

The main corrected run already solved the fatal problems: exact test-image contamination is zero,
the rigorous arm has zero train/test patient overlap, validation is separate from test data, the
paired arms have equal size and binary class counts, and patient-cluster uncertainty is reported.

Three smaller issues remain worth fixing before the manuscript is locked:

1. **Binary-class matching is not enough for NIH ChestX-ray14.**  
   `Effusion absent` includes `No Finding` and many other pathologies; Effusion-positive images can
   also contain co-findings. The patch therefore matches each controlled-leak replacement on the
   **entire normalized Finding Labels profile + AP/PA view position**, not only on the binary target.
   Rare leak candidates without an exact safe match are simply not introduced.

2. **Operating-point metrics need a validation-locked threshold.**  
   The main run used a fixed 0.5 threshold. Its AUC was nearly unchanged while sensitivity and
   specificity shifted strongly, which makes threshold selection important. The patch selects one
   threshold per fold/model/arm on the **validation set only** using Youden's J and then locks that
   threshold before test evaluation. Fixed-0.5 metrics are still exported as supplementary results.

3. **Final deterministic reproducibility + corrected XAI.**  
   The paired models are regenerated with strict deterministic settings. Quantitative XAI is limited
   to ResNet-50 and a preselected ~600-image OOF subset to minimize computation. Rather than treating
   chest-image border activation as "non-clinical" (pleural effusion can genuinely be peripheral),
   the patch quantifies **paired Grad-CAM map similarity and centroid displacement** between rigorous
   and controlled-leak models.

## Minimal-compute policy

- reuse the existing primary cohort and fixed primary fold manifests;
- rerun only the final paired models needed for the corrected controlled-leak comparison;
- rerun the single native-prevalence sensitivity pair because its leak-training composition also
  changes under phenotype/view matching;
- Grad-CAM only for ResNet-50, only on a fixed subset, immediately after each model is trained;
- no LIME/SHAP, no new architecture, no new dataset.

## Kaggle inputs

Attach:

1. the **output of the executed `nih-x-ray-improved-final.ipynb` notebook**  
   (either the `nih_corrected_leakage_audit/` folder or
   `NIH_Corrected_Leakage_Audit_Results.zip`);
2. the same raw **NIH ChestX-ray14** dataset containing the 112,120 radiographs.

Enable a GPU and **Run All** once.

Final archive:

`/kaggle/working/NIH_FinalPatch_Results.zip`

In [1]:
# =========================
# 1. Strict deterministic environment + configuration
# =========================
import os
os.environ["PYTHONHASHSEED"] = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

from pathlib import Path
import re, sys, json, math, time, random, hashlib, shutil, zipfile, platform, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T
import torchvision.models as tvm

from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef, recall_score,
    brier_score_loss, log_loss, roc_curve
)
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold

SEED = 42
FOLDS = 5
VAL_FOLDS = 5

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
EPOCHS = 12
PATIENCE = 3
LR = 1e-4
WEIGHT_DECAY = 1e-4
PRETRAINED = True
LEAK_FRACTION = 1.0

RUN_ARCHITECTURES = ["ResNet50", "ConvNeXtTiny"]

BOOTSTRAPS = 2000
PERMUTATIONS = 2000

# Keep the low-cost native-prevalence sensitivity experiment from the main run,
# but rebuild its controlled-leak arm with the final phenotype/view matching.
RUN_NATIVE_PATCH = True
NATIVE_MODEL = "ResNet50"

# Corrected XAI: ResNet-50 only, preselected OOF subset.
XAI_ARCHITECTURE = "ResNet50"
XAI_MAX_PER_FOLD = 120
XAI_VISUAL_TOTAL = 16

# Usually leave these blank.
PRIOR_OUTPUT_OVERRIDE = ""
DATA_ROOT_OVERRIDE = ""

BASE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/mnt/data")
OUT = BASE / "nih_final_patch"
PRED_DIR = OUT / "predictions"
SPLIT_DIR = OUT / "splits"
XAI_DIR = OUT / "xai"
FIG_DIR = OUT / "figures"
for p in [OUT, PRED_DIR, SPLIT_DIR, XAI_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything()

# Strict deterministic algorithms. CUBLAS_WORKSPACE_CONFIG is set before torch import.
torch.use_deterministic_algorithms(True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CPU execution will be very slow. Use a Kaggle/Colab GPU.")

versions = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda": torch.version.cuda,
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "cudnn_deterministic": bool(torch.backends.cudnn.deterministic),
    "cudnn_benchmark": bool(torch.backends.cudnn.benchmark),
    "deterministic_algorithms": bool(torch.are_deterministic_algorithms_enabled()),
    "CUBLAS_WORKSPACE_CONFIG": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
}
print(json.dumps(versions, indent=2))

Device: cuda
GPU: Tesla T4
{
  "python": "3.13.15",
  "torch": "2.11.0+cu128",
  "torchvision": "0.26.0+cu128",
  "cuda": "12.8",
  "device": "Tesla T4",
  "cudnn_deterministic": true,
  "cudnn_benchmark": false,
  "deterministic_algorithms": true,
  "CUBLAS_WORKSPACE_CONFIG": ":4096:8"
}


In [2]:
# =========================
# 2. Locate the executed main-run output
# =========================
REQUIRED_PRIOR = [
    "primary_cohort_manifest.csv",
    "splits/primary_fold1_rig_train.csv",
    "splits/primary_fold1_val.csv",
    "splits/primary_fold1_test.csv",
]

def valid_prior_root(p):
    p = Path(p)
    return p.exists() and all((p / rel).exists() for rel in REQUIRED_PRIOR)

def discover_prior():
    if PRIOR_OUTPUT_OVERRIDE:
        p = Path(PRIOR_OUTPUT_OVERRIDE)
        if valid_prior_root(p):
            return p
        raise FileNotFoundError(f"Invalid PRIOR_OUTPUT_OVERRIDE: {p}")

    candidates = []
    for base in [Path("/kaggle/input"), Path("/mnt/data"), Path.cwd()]:
        if not base.exists():
            continue

        for p in base.rglob("primary_cohort_manifest.csv"):
            root = p.parent
            if valid_prior_root(root):
                candidates.append(root)

        for z in base.rglob("NIH_Corrected_Leakage_Audit_Results.zip"):
            extract_root = BASE / "_prior_nih_results"
            if extract_root.exists():
                shutil.rmtree(extract_root)
            extract_root.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(z, "r") as fh:
                fh.extractall(extract_root)
            for p in extract_root.rglob("primary_cohort_manifest.csv"):
                root = p.parent
                if valid_prior_root(root):
                    candidates.append(root)

    uniq, seen = [], set()
    for p in candidates:
        rp = str(p.resolve())
        if rp not in seen:
            uniq.append(p)
            seen.add(rp)

    if not uniq:
        raise FileNotFoundError(
            "Executed NIH main-run output not found. Add the OUTPUT of "
            "`nih-x-ray-improved-final.ipynb` as a Kaggle input. It must expose either "
            "`nih_corrected_leakage_audit/` or `NIH_Corrected_Leakage_Audit_Results.zip`."
        )

    uniq.sort(key=lambda p: (0 if p.name == "nih_corrected_leakage_audit" else 1, len(str(p))))
    print("Prior-output candidates:")
    for p in uniq[:10]:
        print(" ", p)
    return uniq[0]

PRIOR = discover_prior()
print("Using prior output:", PRIOR)

cohort = pd.read_csv(PRIOR / "primary_cohort_manifest.csv")
required = {
    "row_id", "Patient_ID", "Label", "Image Index", "Finding Labels",
    "View_Position", "raw_sha256", "decoded_fp"
}
missing = required - set(cohort.columns)
if missing:
    raise RuntimeError(f"Prior primary cohort manifest lacks columns: {sorted(missing)}")

cohort["row_id"] = cohort.row_id.astype(int)
cohort["Patient_ID"] = cohort.Patient_ID.astype(str)
cohort["Label"] = cohort.Label.astype(int)
cohort["View_Position"] = cohort.View_Position.astype(str).str.upper()

print(f"Frozen primary cohort: {len(cohort)} images | {cohort.Patient_ID.nunique()} patients")
print(cohort.Label.value_counts().sort_index().to_string())

Prior-output candidates:
  /kaggle/working/_prior_nih_results/nih_corrected_leakage_audit
  /kaggle/input/notebooks/abrarjavedsorafi/nih-x-ray/nih_corrected_leakage_audit
Using prior output: /kaggle/working/_prior_nih_results/nih_corrected_leakage_audit
Frozen primary cohort: 6000 images | 3911 patients
Label
0    3000
1    3000


In [3]:
# =========================
# 3. Locate raw NIH images and remap stale paths
# =========================
IMG_EXTS = {".png", ".jpg", ".jpeg"}

def resolve_image_root():
    if DATA_ROOT_OVERRIDE:
        p = Path(DATA_ROOT_OVERRIDE)
        if not p.exists():
            raise FileNotFoundError(DATA_ROOT_OVERRIDE)
        return p

    probe = cohort["Image Index"].astype(str).head(50).tolist()
    candidates = []

    for base in [Path("/kaggle/input"), Path("/mnt/data"), Path.cwd()]:
        if not base.exists():
            continue

        # Candidate directories containing many NIH-style image filenames.
        # We only need the chosen root for a one-time filename index.
        for p in [base] + [x for x in base.iterdir() if x.is_dir()]:
            try:
                files = [
                    f for f in p.rglob("*")
                    if f.is_file() and f.suffix.lower() in IMG_EXTS
                ]
            except Exception:
                continue
            if len(files) < 1000:
                continue
            names = {f.name for f in files}
            hits = sum(x in names for x in probe)
            frac = hits / max(len(probe), 1)
            if frac > 0:
                candidates.append((frac, len(files), p, files))
            if frac >= 0.95:
                break
        if candidates and max(x[0] for x in candidates) >= 0.95:
            break

    if not candidates:
        raise FileNotFoundError(
            "Raw NIH ChestX-ray14 images were not found. Attach the same NIH dataset used "
            "by the main run."
        )

    candidates.sort(key=lambda x: (-x[0], -x[1], len(str(x[2]))))
    frac, n, root, files = candidates[0]
    if frac < 0.80:
        raise RuntimeError(f"Best NIH image-root probe match is only {frac:.1%}: {root}")
    print(f"Chosen image root: {root} | files={n:,} | probe={frac:.1%}")
    return root, files

ROOT, image_files = resolve_image_root()

image_index = {}
for f in sorted(image_files, key=lambda x: (len(str(x)), str(x))):
    image_index.setdefault(f.name, str(f))

missing = [x for x in cohort["Image Index"].astype(str) if x not in image_index]
if missing:
    raise RuntimeError(f"{len(missing)} frozen cohort images are absent from raw data: {missing[:5]}")

cohort["filepath"] = cohort["Image Index"].astype(str).map(image_index)
print("All primary cohort rows remapped to raw image files.")

Chosen image root: /kaggle/input | files=112,128 | probe=100.0%
All primary cohort rows remapped to raw image files.


In [4]:
# =========================
# 4. NIH-specific phenotype/view matching variables
# =========================
def normalize_finding_profile(x):
    labels = [t.strip() for t in str(x).split("|") if t.strip()]
    return "|".join(sorted(set(labels)))

def parse_age(x):
    if pd.isna(x):
        return np.nan
    m = re.search(r"(\d+)", str(x))
    return float(m.group(1)) if m else np.nan

def enrich(df):
    x = df.copy()
    x["row_id"] = x.row_id.astype(int)
    x["Patient_ID"] = x.Patient_ID.astype(str)
    x["Label"] = x.Label.astype(int)
    x["View_Position"] = x.View_Position.astype(str).str.upper()
    x["FindingProfile"] = x["Finding Labels"].map(normalize_finding_profile)
    # Mandatory exact matching stratum: complete multi-label profile + acquisition view.
    x["MatchStratum"] = x["FindingProfile"] + "||VIEW=" + x["View_Position"]

    if "Patient Gender" in x.columns:
        x["GenderNorm"] = x["Patient Gender"].astype(str).str.upper()
    else:
        x["GenderNorm"] = "UNKNOWN"

    if "Patient Age" in x.columns:
        x["AgeNum"] = x["Patient Age"].map(parse_age)
    else:
        x["AgeNum"] = np.nan

    x["filepath"] = x["Image Index"].astype(str).map(image_index)
    if x.filepath.isna().any():
        raise RuntimeError("A manifest contains images absent from the attached raw dataset.")
    return x

cohort = enrich(cohort)

print("Top phenotype/view strata:")
print(cohort.MatchStratum.value_counts().head(15).to_string())

# A reviewer-friendly description of the heterogeneous binary negative class.
negative = cohort[cohort.Label == 0]
print(
    f"\nEffusion-absent images: {len(negative)}; "
    f"No Finding: {(negative.FindingProfile == 'No Finding').sum()} "
    f"({100*(negative.FindingProfile == 'No Finding').mean():.1f}%)"
)

Top phenotype/view strata:
MatchStratum
No Finding||VIEW=PA                           1183
No Finding||VIEW=AP                            662
Effusion||VIEW=PA                              470
Effusion||VIEW=AP                              446
Effusion|Infiltration||VIEW=AP                 261
Infiltration||VIEW=PA                          155
Infiltration||VIEW=AP                          145
Atelectasis|Effusion||VIEW=PA                  131
Atelectasis|Effusion||VIEW=AP                  126
Effusion|Infiltration||VIEW=PA                 106
Atelectasis|Effusion|Infiltration||VIEW=AP     100
Atelectasis||VIEW=PA                            75
Atelectasis|Effusion|Infiltration||VIEW=PA      68
Effusion|Mass||VIEW=PA                          64
Effusion|Pneumothorax||VIEW=PA                  64

Effusion-absent images: 3000; No Finding: 1845 (61.5%)


In [5]:
# =========================
# 5. Rebuild ONLY the controlled-leak primary train manifests
#    with exact full-phenotype + AP/PA view matching
# =========================
def remap_manifest(path):
    return enrich(pd.read_csv(path))

def choose_removals_greedy(base, chosen_candidates):
    """
    All rows entering here already share the exact FindingProfile + View_Position.
    To reduce residual demographic drift, choose a removable safe row with the same
    gender when possible and the closest available age. This is a secondary matching
    preference; phenotype/view matching is the hard constraint.
    """
    available = base.copy()
    removed = []

    for _, c in chosen_candidates.sort_values("row_id").iterrows():
        pool = available
        same_gender = pool[pool.GenderNorm == c.GenderNorm]
        if len(same_gender):
            pool = same_gender

        if np.isfinite(c.AgeNum) and pool.AgeNum.notna().any():
            distance = (pool.AgeNum - c.AgeNum).abs()
            idx = distance.sort_values(kind="mergesort").index[0]
        else:
            idx = pool.sort_values("row_id").index[0]

        removed.append(int(available.loc[idx, "row_id"]))
        available = available.drop(index=idx)

    return removed

def build_matched_leak_train(frozen_cohort, rig, val, test, fold):
    test_ids = set(test.row_id.astype(int))
    test_patients = set(test.Patient_ID.astype(str))
    test_raw = set(test.raw_sha256.astype(str))
    test_dec = set(test.decoded_fp.astype(str))

    # These are the only eligible same-patient siblings from the frozen outer-training pool.
    candidates = frozen_cohort[
        (~frozen_cohort.row_id.astype(int).isin(test_ids))
        & (frozen_cohort.Patient_ID.astype(str).isin(test_patients))
        & (~frozen_cohort.raw_sha256.astype(str).isin(test_raw))
        & (~frozen_cohort.decoded_fp.astype(str).isin(test_dec))
    ].copy()

    leak_parts = []
    remove_ids = []
    used_candidate_ids = []
    unmatched_candidates = 0

    for stratum, cg in candidates.groupby("MatchStratum", sort=True):
        bg = rig[rig.MatchStratum == stratum].copy()
        if len(bg) == 0:
            unmatched_candidates += len(cg)
            continue

        n = min(int(round(len(cg) * LEAK_FRACTION)), len(cg), len(bg))
        if n <= 0:
            unmatched_candidates += len(cg)
            continue

        # Deterministic candidate choice; all eligible candidates are used until safe matches run out.
        chosen = cg.sort_values("row_id").head(n).copy()
        removed = choose_removals_greedy(bg, chosen)

        leak_parts.append(chosen)
        remove_ids.extend(removed)
        used_candidate_ids.extend(chosen.row_id.astype(int).tolist())
        unmatched_candidates += len(cg) - n

    leak = rig[~rig.row_id.astype(int).isin(remove_ids)].copy()
    if leak_parts:
        leak = pd.concat([leak] + leak_parts, ignore_index=True)
    leak = leak.sample(frac=1, random_state=SEED + 10000*fold + 999).reset_index(drop=True)

    def ids(x): return set(x.row_id.astype(int))
    def pats(x): return set(x.Patient_ID.astype(str))
    def raws(x): return set(x.raw_sha256.astype(str))
    def decs(x): return set(x.decoded_fp.astype(str))

    assert ids(test).isdisjoint(ids(rig))
    assert ids(test).isdisjoint(ids(leak))
    assert raws(test).isdisjoint(raws(rig))
    assert raws(test).isdisjoint(raws(leak))
    assert decs(test).isdisjoint(decs(rig))
    assert decs(test).isdisjoint(decs(leak))
    assert pats(test).isdisjoint(pats(rig))
    assert pats(test).isdisjoint(pats(val))
    assert pats(val).isdisjoint(pats(rig))
    assert pats(val).isdisjoint(pats(leak))
    assert len(rig) == len(leak)

    # Core final-patch assertion: the entire multi-label profile + view distribution is exact.
    rig_strata = rig.MatchStratum.value_counts().sort_index()
    leak_strata = leak.MatchStratum.value_counts().sort_index()
    assert rig_strata.equals(leak_strata), "Full phenotype/view training composition differs."

    # The exact 14-label marginal counts are consequently preserved too.
    all_labels = sorted({
        lab
        for s in frozen_cohort.FindingProfile
        for lab in s.split("|")
        if lab and lab != "No Finding"
    })
    for lab in all_labels:
        rc = int(rig["FindingProfile"].str.split("|").apply(lambda z: lab in z).sum())
        lc = int(leak["FindingProfile"].str.split("|").apply(lambda z: lab in z).sum())
        assert rc == lc, f"Marginal count mismatch for {lab}: {rc} vs {lc}"

    # Exposure values for test rows.
    sib = leak.Patient_ID.astype(str).value_counts()
    same_profile = leak.groupby(["Patient_ID", "FindingProfile"]).size()
    same_view = leak.groupby(["Patient_ID", "View_Position"]).size()

    test = test.copy()
    test["n_train_siblings"] = test.Patient_ID.astype(str).map(sib).fillna(0).astype(int)
    test["n_same_profile_siblings"] = [
        int(same_profile.get((str(p), str(fp)), 0))
        for p, fp in zip(test.Patient_ID, test.FindingProfile)
    ]
    test["n_same_view_siblings"] = [
        int(same_view.get((str(p), str(v)), 0))
        for p, v in zip(test.Patient_ID, test.View_Position)
    ]

    def age_mean(x):
        return float(x.AgeNum.dropna().mean()) if x.AgeNum.notna().any() else np.nan

    def age_smd(a, b):
        aa, bb = a.AgeNum.dropna(), b.AgeNum.dropna()
        if len(aa) < 2 or len(bb) < 2:
            return np.nan
        pooled = np.sqrt((aa.var(ddof=1) + bb.var(ddof=1))/2)
        return float((bb.mean()-aa.mean())/pooled) if pooled > 0 else 0.0

    def female_pct(x):
        if not (x.GenderNorm.isin(["F","M"]).any()):
            return np.nan
        return float((x.GenderNorm=="F").mean())

    audit = {
        "fold": fold,
        "test_n": len(test),
        "test_patients": test.Patient_ID.nunique(),
        "rig_train_n": len(rig),
        "leak_train_n": len(leak),
        "val_n": len(val),
        "eligible_sibling_candidates": len(candidates),
        "used_sibling_candidates": len(used_candidate_ids),
        "candidate_utilization_pct": 100*len(used_candidate_ids)/max(len(candidates),1),
        "unmatched_candidates": unmatched_candidates,
        "rig_train_test_patient_overlap": len(pats(rig)&pats(test)),
        "leak_train_test_patient_overlap": len(pats(leak)&pats(test)),
        "rig_train_test_rawhash_overlap": len(raws(rig)&raws(test)),
        "leak_train_test_rawhash_overlap": len(raws(leak)&raws(test)),
        "rig_train_test_decoded_overlap": len(decs(rig)&decs(test)),
        "leak_train_test_decoded_overlap": len(decs(leak)&decs(test)),
        "test_exposure_pct": float((test.n_train_siblings>0).mean()),
        "same_profile_exposure_pct": float((test.n_same_profile_siblings>0).mean()),
        "same_view_exposure_pct": float((test.n_same_view_siblings>0).mean()),
        "max_siblings": int(test.n_train_siblings.max()),
        "exact_match_strata_preserved": bool(rig_strata.equals(leak_strata)),
        "rig_age_mean": age_mean(rig),
        "leak_age_mean": age_mean(leak),
        "age_smd_leak_minus_rig": age_smd(rig, leak),
        "rig_female_pct": female_pct(rig),
        "leak_female_pct": female_pct(leak),
    }
    if np.isfinite(audit["rig_female_pct"]) and np.isfinite(audit["leak_female_pct"]):
        audit["female_pct_diff"] = audit["leak_female_pct"] - audit["rig_female_pct"]
    else:
        audit["female_pct_diff"] = np.nan

    return leak, test, audit

FOLD_DATA = {}
audit_rows = []

for fold in range(1, FOLDS+1):
    rig = remap_manifest(PRIOR / "splits" / f"primary_fold{fold}_rig_train.csv")
    val = remap_manifest(PRIOR / "splits" / f"primary_fold{fold}_val.csv")
    test = remap_manifest(PRIOR / "splits" / f"primary_fold{fold}_test.csv")

    leak, test, audit = build_matched_leak_train(cohort, rig, val, test, fold)

    FOLD_DATA[fold] = {"rig":rig, "leak":leak, "val":val, "test":test}
    audit_rows.append(audit)

    for role, frame in [("rig_train",rig),("leak_train",leak),("val",val),("test",test)]:
        x = frame.copy()
        x["fold"] = fold
        x["role"] = role
        x.to_csv(SPLIT_DIR / f"primary_fold{fold}_{role}.csv", index=False)

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(OUT / "phenotype_view_matched_leakage_audit.csv", index=False)

print(audit_df.to_string(index=False))
assert audit_df.exact_match_strata_preserved.all()
assert audit_df.rig_train_test_patient_overlap.eq(0).all()
assert audit_df.rig_train_test_rawhash_overlap.eq(0).all()
assert audit_df.leak_train_test_rawhash_overlap.eq(0).all()
assert audit_df.rig_train_test_decoded_overlap.eq(0).all()
assert audit_df.leak_train_test_decoded_overlap.eq(0).all()

print("\nFULL PHENOTYPE + VIEW MATCHING AUDIT: PASSED")
print(f"Mean test-image sibling exposure: {100*audit_df.test_exposure_pct.mean():.1f}%")
print(f"Mean same-profile exposure: {100*audit_df.same_profile_exposure_pct.mean():.1f}%")
if 100*audit_df.test_exposure_pct.mean() < 20:
    print("WARNING: controlled exposure fell below 20%; interpret effect estimates as low-exposure.")

 fold  test_n  test_patients  rig_train_n  leak_train_n  val_n  eligible_sibling_candidates  used_sibling_candidates  candidate_utilization_pct  unmatched_candidates  rig_train_test_patient_overlap  leak_train_test_patient_overlap  rig_train_test_rawhash_overlap  leak_train_test_rawhash_overlap  rig_train_test_decoded_overlap  leak_train_test_decoded_overlap  test_exposure_pct  same_profile_exposure_pct  same_view_exposure_pct  max_siblings  exact_match_strata_preserved  rig_age_mean  leak_age_mean  age_smd_leak_minus_rig  rig_female_pct  leak_female_pct  female_pct_diff
    1    1200           1045         2982          2982    742                         1076                      964                  89.591078                   112                               0                              448                               0                                0                               0                                0           0.478333                   0.142500                

In [6]:
# =========================
# 6. Prediction-independent XAI subset and visual-case preselection
# =========================
def stratified_xai_sample(test, fold, max_n=XAI_MAX_PER_FOLD):
    x = test.copy()
    x["XAI_Stratum"] = x.Label.astype(str) + "||" + x.View_Position.astype(str)
    groups = list(x.groupby("XAI_Stratum", sort=True))
    if not groups:
        return x.iloc[0:0].copy()

    base_quota = max(1, max_n // len(groups))
    parts, used = [], set()

    for gi, (_, g) in enumerate(groups):
        n = min(base_quota, len(g))
        s = g.sample(n=n, random_state=SEED + 1000*fold + gi)
        parts.append(s)
        used.update(s.row_id.astype(int))

    out = pd.concat(parts, ignore_index=True) if parts else x.iloc[0:0].copy()
    remaining_n = max_n - len(out)
    if remaining_n > 0:
        rem = x[~x.row_id.astype(int).isin(used)]
        if len(rem):
            extra = rem.sample(
                n=min(remaining_n,len(rem)),
                random_state=SEED + 1000*fold + 999
            )
            out = pd.concat([out,extra], ignore_index=True)

    return out.reset_index(drop=True)

XAI_TEST_BY_FOLD = {}
xai_selection = []

for fold in range(1,FOLDS+1):
    x = stratified_xai_sample(FOLD_DATA[fold]["test"], fold)
    x["fold"] = fold
    XAI_TEST_BY_FOLD[fold] = x
    xai_selection.append(x)

xai_selected = pd.concat(xai_selection, ignore_index=True)
xai_selected[[
    "row_id","Image Index","Patient_ID","Label","FindingProfile",
    "View_Position","fold","n_train_siblings"
]].to_csv(XAI_DIR / "preselected_quantitative_xai_cases.csv", index=False)

# Separate small visual subset, selected before model predictions are available.
vis_parts = []
strata = list(xai_selected.groupby(["Label","View_Position"], sort=True))
per = max(1, XAI_VISUAL_TOTAL // max(len(strata),1))
for i, (_,g) in enumerate(strata):
    n = min(per,len(g))
    vis_parts.append(g.sample(n=n,random_state=SEED+8000+i))
visual_cases = pd.concat(vis_parts,ignore_index=True) if vis_parts else xai_selected.head(0)
if len(visual_cases) < XAI_VISUAL_TOTAL:
    used = set(visual_cases.row_id.astype(int))
    rem = xai_selected[~xai_selected.row_id.astype(int).isin(used)]
    if len(rem):
        extra = rem.sample(
            n=min(XAI_VISUAL_TOTAL-len(visual_cases),len(rem)),
            random_state=SEED+8999
        )
        visual_cases = pd.concat([visual_cases,extra],ignore_index=True)

VISUAL_IDS = set(visual_cases.row_id.astype(int))
visual_cases[[
    "row_id","Image Index","Patient_ID","Label","FindingProfile","View_Position","fold"
]].to_csv(XAI_DIR / "preselected_visual_cases.csv", index=False)

print(f"Quantitative XAI cases: {len(xai_selected)}")
print(f"Visual XAI cases: {len(visual_cases)}")

Quantitative XAI cases: 600
Visual XAI cases: 16


In [7]:
# =========================
# 7. Chest-radiograph data pipeline
# =========================
IM_MEAN = (0.485,0.456,0.406)
IM_STD = (0.229,0.224,0.225)

TRAIN_TF = T.Compose([
    T.Resize((IMG_SIZE,IMG_SIZE)),
    T.Grayscale(num_output_channels=3),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomAffine(degrees=7,translate=(0.04,0.04)),
    T.ColorJitter(brightness=0.10,contrast=0.10),
    T.ToTensor(),
    T.Normalize(IM_MEAN,IM_STD),
])

EVAL_TF = T.Compose([
    T.Resize((IMG_SIZE,IMG_SIZE)),
    T.Grayscale(num_output_channels=3),
    T.ToTensor(),
    T.Normalize(IM_MEAN,IM_STD),
])

class NIHDataset(Dataset):
    def __init__(self,df,train=False):
        self.df=df.reset_index(drop=True)
        self.tf=TRAIN_TF if train else EVAL_TF
    def __len__(self):
        return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        with Image.open(r.filepath) as im:
            im=im.convert("L")
            x=self.tf(im)
        return x,int(r.Label),int(r.row_id),str(r.Patient_ID)

def make_loader(df,train=False,seed=SEED):
    g=torch.Generator()
    g.manual_seed(seed)
    return DataLoader(
        NIHDataset(df,train=train),
        batch_size=BATCH_SIZE,
        shuffle=train,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type=="cuda"),
        generator=g if train else None,
        persistent_workers=False,
    )

print("Data pipeline ready.")

Data pipeline ready.


In [8]:
# =========================
# 8. Models, validation-locked threshold, deterministic training
# =========================
def make_resnet50():
    w=tvm.ResNet50_Weights.DEFAULT if PRETRAINED else None
    m=tvm.resnet50(weights=w)
    m.fc=nn.Linear(m.fc.in_features,2)
    return m

def make_convnext_tiny():
    w=tvm.ConvNeXt_Tiny_Weights.DEFAULT if PRETRAINED else None
    m=tvm.convnext_tiny(weights=w)
    m.classifier[2]=nn.Linear(m.classifier[2].in_features,2)
    return m

MODEL_FACTORIES={"ResNet50":make_resnet50,"ConvNeXtTiny":make_convnext_tiny}

for name in RUN_ARCHITECTURES:
    try:
        m=MODEL_FACTORIES[name]()
        del m
    except Exception as e:
        raise RuntimeError(
            f"Could not initialize ImageNet weights for {name}. Enable Internet for the "
            f"first Kaggle run or provide cached torchvision weights. {repr(e)}"
        )
print("Pretrained-weight preflight: PASSED")

def autocast_ctx():
    return torch.autocast(device_type="cuda",dtype=torch.float16,enabled=USE_AMP)

@torch.no_grad()
def predict_prob(model,loader):
    rows=[]
    model.eval()
    for x,y,row_id,patient in loader:
        x=x.to(DEVICE,non_blocking=True)
        with autocast_ctx():
            logits=model(x)
        p=torch.softmax(logits.float(),dim=1)[:,1].cpu().numpy()
        for yy,pp,rid,pat in zip(y.numpy(),p,row_id.numpy(),patient):
            rows.append({
                "row_id":int(rid),"Patient_ID":str(pat),
                "Label":int(yy),"prob":float(pp)
            })
    return pd.DataFrame(rows)

def choose_youden_threshold(y,p):
    y=np.asarray(y,dtype=int)
    p=np.asarray(p,dtype=float)
    if len(np.unique(y))<2:
        return 0.5
    fpr,tpr,thr=roc_curve(y,p)
    finite=np.isfinite(thr)
    fpr,tpr,thr=fpr[finite],tpr[finite],thr[finite]
    if len(thr)==0:
        return 0.5
    j=tpr-fpr
    best=np.flatnonzero(j==np.nanmax(j))
    # Deterministic tie break: threshold closest to 0.5.
    idx=best[np.argmin(np.abs(thr[best]-0.5))]
    return float(np.clip(thr[idx],0.0,1.0))

def class_weights(df):
    c=df.Label.value_counts()
    n=len(df)
    return torch.tensor([
        n/(2*max(int(c.get(0,1)),1)),
        n/(2*max(int(c.get(1,1)),1))
    ],dtype=torch.float32,device=DEVICE)

def train_one(model_name,train_df,val_df,fold,arm,class_weight=None,experiment="primary"):
    model_offset=0 if model_name=="ResNet50" else 10
    exp_offset=0 if experiment=="primary" else 10000
    run_seed=SEED+fold*100+model_offset+exp_offset
    seed_everything(run_seed)

    model=MODEL_FACTORIES[model_name]().to(DEVICE)
    trL=make_loader(train_df,train=True,seed=run_seed)
    vaL=make_loader(val_df,train=False,seed=run_seed)

    opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
    loss_fn=nn.CrossEntropyLoss(weight=class_weight)

    scaler=torch.amp.GradScaler("cuda",enabled=USE_AMP) if hasattr(torch,"amp") else torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_auc=-np.inf
    best_state=None
    best_epoch=None
    wait=0
    hist=[]

    print(f"{experiment} | {model_name} | fold {fold} | {arm} | train={len(train_df)} val={len(val_df)}")

    for ep in range(1,EPOCHS+1):
        t0=time.time()
        model.train()
        total=0.0
        n=0

        for x,y,_,_ in trL:
            x=x.to(DEVICE,non_blocking=True)
            y=y.to(DEVICE,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast_ctx():
                logits=model(x)
                loss=loss_fn(logits,y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            total+=float(loss.detach())*len(y)
            n+=len(y)

        sch.step()
        vp=predict_prob(model,vaL)
        vauc=roc_auc_score(vp.Label,vp.prob) if vp.Label.nunique()==2 else np.nan
        hist.append({"epoch":ep,"train_loss":total/max(n,1),"val_auc":vauc})
        print(f"  epoch {ep:02d}/{EPOCHS} loss={total/max(n,1):.4f} valAUC={vauc:.4f} time={time.time()-t0:.0f}s")

        if np.isfinite(vauc) and vauc>best_auc+1e-6:
            best_auc=vauc
            best_epoch=ep
            wait=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            wait+=1
            if wait>=PATIENCE:
                print("  early stop")
                break

    if best_state is None:
        raise RuntimeError(f"No valid checkpoint: {experiment} {model_name} fold {fold} {arm}")

    model.load_state_dict(best_state)
    model.eval()

    # Lock the operating threshold using VALIDATION ONLY.
    vp=predict_prob(model,make_loader(val_df,train=False,seed=run_seed))
    locked_threshold=choose_youden_threshold(vp.Label,vp.prob)

    return model,pd.DataFrame(hist),int(best_epoch),float(locked_threshold),vp

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 225MB/s]


Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 188MB/s]


Pretrained-weight preflight: PASSED


In [9]:
# =========================
# 9. Dependency-free ResNet-50 Grad-CAM
# =========================
def gradcam_native(model,x,targets):
    """
    Grad-CAM at ResNet-50 layer4 native spatial resolution.
    The TRUE binary class is used as the target for both paired models.
    """
    act={}
    grad={}
    layer=model.layer4

    def fh(m,inp,out):
        act["v"]=out
    def bh(m,gin,gout):
        grad["v"]=gout[0]

    h1=layer.register_forward_hook(fh)
    h2=layer.register_full_backward_hook(bh)

    model.zero_grad(set_to_none=True)
    logits=model(x)
    score=logits.gather(1,targets.view(-1,1)).sum()
    score.backward()

    A=act["v"]
    G=grad["v"]
    w=G.mean(dim=(2,3),keepdim=True)
    cam=torch.relu((w*A).sum(dim=1))

    flat=cam.flatten(1)
    mn=flat.min(dim=1).values[:,None,None]
    mx=flat.max(dim=1).values[:,None,None]
    cam=(cam-mn)/(mx-mn+1e-8)

    h1.remove(); h2.remove()
    return cam.detach()

def cam_centroid(cam):
    H,W=cam.shape
    mass=float(cam.sum())+1e-12
    yy,xx=np.mgrid[0:H,0:W]
    cx=float((cam*xx).sum()/mass)/(max(W-1,1))
    cy=float((cam*yy).sum()/mass)/(max(H-1,1))
    return cx,cy

def cam_pair_metrics(a,b):
    af=a.reshape(-1).astype(float)
    bf=b.reshape(-1).astype(float)
    denom=np.linalg.norm(af)*np.linalg.norm(bf)
    cosine=float(np.dot(af,bf)/denom) if denom>0 else np.nan

    if np.std(af)>0 and np.std(bf)>0:
        corr=float(np.corrcoef(af,bf)[0,1])
    else:
        corr=np.nan

    ax,ay=cam_centroid(a)
    bx,by=cam_centroid(b)
    shift=float(np.sqrt((ax-bx)**2+(ay-by)**2)/np.sqrt(2))
    return cosine,corr,shift

def denormalize(x):
    mean=torch.tensor(IM_MEAN,device=x.device).view(3,1,1)
    std=torch.tensor(IM_STD,device=x.device).view(3,1,1)
    return torch.clamp(x*std+mean,0,1)

def save_overlay(x,cam,path,title):
    base=denormalize(x.detach()).cpu().permute(1,2,0).numpy().mean(axis=2)
    c=torch.tensor(cam)[None,None]
    c=torch.nn.functional.interpolate(c,size=(IMG_SIZE,IMG_SIZE),mode="bilinear",align_corners=False)[0,0].numpy()
    fig,ax=plt.subplots(figsize=(4,4))
    ax.imshow(base,cmap="gray")
    ax.imshow(c,cmap="jet",alpha=0.42,vmin=0,vmax=1)
    ax.axis("off")
    ax.set_title(title,fontsize=8)
    fig.tight_layout()
    fig.savefig(path,dpi=200,bbox_inches="tight")
    plt.close(fig)

def extract_xai(model,xai_df,fold,arm):
    loader=make_loader(xai_df,train=False,seed=SEED+fold)
    cams={}
    meta={}
    model.eval()

    for x,y,row_id,patient in loader:
        x=x.to(DEVICE)
        y=y.to(DEVICE)

        # Full precision for stable attribution gradients.
        c=gradcam_native(model,x,y).cpu().numpy()

        for j,rid in enumerate(row_id.numpy()):
            rid=int(rid)
            cams[rid]=c[j].astype(np.float32)
            rr=xai_df[xai_df.row_id.astype(int)==rid].iloc[0]
            meta[rid]={
                "Patient_ID":str(rr.Patient_ID),
                "Label":int(rr.Label),
                "FindingProfile":str(rr.FindingProfile),
                "View_Position":str(rr.View_Position),
                "n_train_siblings":int(rr.n_train_siblings),
            }

            if rid in VISUAL_IDS:
                save_overlay(
                    x[j],c[j],
                    XAI_DIR/f"row{rid}_fold{fold}_{arm}.png",
                    f"fold {fold} | {arm} | y={int(rr.Label)} | {rr.View_Position}"
                )

    return cams,meta

In [10]:
# =========================
# 10. Final primary deterministic paired rerun + in-run corrected XAI
# =========================
all_oof=[]
training_records=[]
xai_pair_rows=[]

for model_name in RUN_ARCHITECTURES:
    print("\n"+"="*90)
    print("FINAL PRIMARY ARCHITECTURE:",model_name)
    print("="*90)

    for fold in range(1,FOLDS+1):
        D=FOLD_DATA[fold]
        test=D["test"].copy()
        arm_predictions={}
        arm_thresholds={}
        rig_cams=None
        rig_meta=None

        for arm,key in [("rigorous","rig"),("controlled_leak","leak")]:
            model,hist,best_epoch,threshold,val_pred=train_one(
                model_name,D[key],D["val"],fold,arm,
                class_weight=None,experiment="primary"
            )

            pp=predict_prob(model,make_loader(test,train=False,seed=SEED+fold))
            pp["pred05"]=(pp.prob>=0.5).astype(int)
            pp["pred_locked"]=(pp.prob>=threshold).astype(int)
            pp["locked_threshold"]=threshold

            pp.to_csv(PRED_DIR/f"primary_{model_name}_fold{fold}_{arm}.csv",index=False)
            hist.to_csv(PRED_DIR/f"primary_{model_name}_fold{fold}_{arm}_history.csv",index=False)
            val_pred.assign(
                locked_threshold=threshold,
                pred_locked=(val_pred.prob>=threshold).astype(int)
            ).to_csv(
                PRED_DIR/f"primary_{model_name}_fold{fold}_{arm}_validation_predictions.csv",
                index=False
            )

            arm_predictions[arm]=pp
            arm_thresholds[arm]=threshold
            training_records.append({
                "experiment":"primary","architecture":model_name,"fold":fold,"arm":arm,
                "best_epoch":best_epoch,"locked_threshold":threshold,
                "train_n":len(D[key]),"val_n":len(D["val"]),"test_n":len(test)
            })

            # Corrected XAI is deliberately limited to ResNet-50.
            if model_name==XAI_ARCHITECTURE:
                cams,meta=extract_xai(model,XAI_TEST_BY_FOLD[fold],fold,arm)
                if arm=="rigorous":
                    rig_cams,rig_meta=cams,meta
                else:
                    common=sorted(set(rig_cams)&set(cams))
                    for rid in common:
                        cos,corr,shift=cam_pair_metrics(rig_cams[rid],cams[rid])
                        mm=meta[rid]
                        xai_pair_rows.append({
                            "row_id":rid,
                            "fold":fold,
                            "Patient_ID":mm["Patient_ID"],
                            "Label":mm["Label"],
                            "FindingProfile":mm["FindingProfile"],
                            "View_Position":mm["View_Position"],
                            "n_train_siblings":mm["n_train_siblings"],
                            "exposed":int(mm["n_train_siblings"]>0),
                            "cam_cosine_similarity":cos,
                            "cam_pearson_r":corr,
                            "cam_centroid_shift":shift,
                        })

            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        r=arm_predictions["rigorous"].rename(
            columns={
                "prob":"prob_rig","pred05":"pred05_rig","pred_locked":"pred_locked_rig",
                "locked_threshold":"threshold_rig"
            }
        )
        l=arm_predictions["controlled_leak"][[
            "row_id","prob","pred05","pred_locked","locked_threshold"
        ]].rename(
            columns={
                "prob":"prob_leak","pred05":"pred05_leak","pred_locked":"pred_locked_leak",
                "locked_threshold":"threshold_leak"
            }
        )

        merged=test[[
            "row_id","Patient_ID","Label","Image Index","FindingProfile","View_Position",
            "n_train_siblings","n_same_profile_siblings","n_same_view_siblings"
        ]].merge(
            r[[
                "row_id","prob_rig","pred05_rig","pred_locked_rig","threshold_rig"
            ]],on="row_id",how="inner"
        ).merge(
            l,on="row_id",how="inner"
        )

        assert len(merged)==len(test)
        merged["fold"]=fold
        merged["architecture"]=model_name
        all_oof.append(merged)

oof=pd.concat(all_oof,ignore_index=True)
oof.to_csv(OUT/"primary_oof_paired_predictions.csv",index=False)

training_log=pd.DataFrame(training_records)
training_log.to_csv(OUT/"training_summary.csv",index=False)

xai_pairs=pd.DataFrame(xai_pair_rows)
xai_pairs.to_csv(XAI_DIR/"gradcam_paired_map_metrics.csv",index=False)

print("\nPrimary OOF rows:",len(oof))
print("XAI paired rows:",len(xai_pairs))


FINAL PRIMARY ARCHITECTURE: ResNet50
primary | ResNet50 | fold 1 | rigorous | train=2982 val=742
  epoch 01/12 loss=0.5837 valAUC=0.8255 time=117s
  epoch 02/12 loss=0.4818 valAUC=0.8420 time=78s
  epoch 03/12 loss=0.4391 valAUC=0.8383 time=78s
  epoch 04/12 loss=0.4115 valAUC=0.8320 time=78s
  epoch 05/12 loss=0.3705 valAUC=0.8340 time=77s
  early stop
primary | ResNet50 | fold 1 | controlled_leak | train=2982 val=742
  epoch 01/12 loss=0.5856 valAUC=0.8087 time=88s
  epoch 02/12 loss=0.4983 valAUC=0.8296 time=78s
  epoch 03/12 loss=0.4479 valAUC=0.8258 time=78s
  epoch 04/12 loss=0.4293 valAUC=0.8252 time=77s
  epoch 05/12 loss=0.3803 valAUC=0.8348 time=79s
  epoch 06/12 loss=0.3295 valAUC=0.8203 time=79s
  epoch 07/12 loss=0.2725 valAUC=0.7847 time=78s
  epoch 08/12 loss=0.2460 valAUC=0.8075 time=78s
  early stop
primary | ResNet50 | fold 2 | rigorous | train=3014 val=752
  epoch 01/12 loss=0.6055 valAUC=0.7826 time=80s
  epoch 02/12 loss=0.5227 valAUC=0.8268 time=79s
  epoch 03/12

In [11]:
# =========================
# 11. Patient-cluster paired statistics
#     Main thresholded metrics use validation-locked thresholds.
# =========================
PROB_METRICS=["AUC","PR_AUC","Brier","LogLoss"]
THRESH_METRICS=["ACC","Macro_F1","Balanced_ACC","MCC","Sensitivity","Specificity"]
ALL_METRICS=PROB_METRICS+THRESH_METRICS

def metric_on_df(d,arm,metric,locked=True):
    y=d.Label.to_numpy(dtype=int)
    p=d[f"prob_{arm}"].to_numpy(dtype=float)

    if metric=="AUC":
        return roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan
    if metric=="PR_AUC":
        return average_precision_score(y,p) if len(np.unique(y))==2 else np.nan
    if metric=="Brier":
        return brier_score_loss(y,p)
    if metric=="LogLoss":
        return log_loss(y,np.c_[1-p,p],labels=[0,1])

    col=f"pred_locked_{arm}" if locked else f"pred05_{arm}"
    pred=d[col].to_numpy(dtype=int)

    if metric=="ACC":
        return accuracy_score(y,pred)
    if metric=="Macro_F1":
        return f1_score(y,pred,average="macro",zero_division=0)
    if metric=="Balanced_ACC":
        return balanced_accuracy_score(y,pred)
    if metric=="MCC":
        return matthews_corrcoef(y,pred)
    if metric=="Sensitivity":
        return recall_score(y,pred,pos_label=1,zero_division=0)
    if metric=="Specificity":
        return recall_score(y,pred,pos_label=0,zero_division=0)
    raise KeyError(metric)

def macro_fold_metric(d,arm,metric,locked=True):
    vals=[]
    for _,g in d.groupby("fold"):
        v=metric_on_df(g,arm,metric,locked=locked)
        if np.isfinite(v):
            vals.append(v)
    return float(np.mean(vals)) if vals else np.nan

def paired_cluster_bootstrap(d,metric,locked=True,B=BOOTSTRAPS,seed=SEED):
    pats=np.array(sorted(d.Patient_ID.astype(str).unique()))
    byp={p:d[d.Patient_ID.astype(str)==p] for p in pats}
    rng=np.random.default_rng(seed)
    deltas=[]

    for _ in range(B):
        samp=rng.choice(pats,size=len(pats),replace=True)
        x=pd.concat([byp[p] for p in samp],ignore_index=True)
        rv=macro_fold_metric(x,"rig",metric,locked)
        lv=macro_fold_metric(x,"leak",metric,locked)
        if np.isfinite(rv) and np.isfinite(lv):
            deltas.append(lv-rv)

    if len(deltas)<max(200,B//2):
        raise RuntimeError(f"Too few valid bootstrap replicates for {metric}")
    return float(np.percentile(deltas,2.5)),float(np.percentile(deltas,97.5)),len(deltas)

def cluster_randomization_p(d,metric,locked=True,B=PERMUTATIONS,seed=SEED+7):
    obs=macro_fold_metric(d,"leak",metric,locked)-macro_fold_metric(d,"rig",metric,locked)
    pats=np.array(sorted(d.Patient_ID.astype(str).unique()))
    rng=np.random.default_rng(seed)
    extreme=0
    valid=0

    for _ in range(B):
        swap=set(pats[rng.integers(0,2,size=len(pats)).astype(bool)])
        x=d.copy()
        mask=x.Patient_ID.astype(str).isin(swap).to_numpy()

        # Swap BOTH probability and thresholded paired outcomes patient-wise.
        for a,b in [
            ("prob_rig","prob_leak"),
            ("pred_locked_rig","pred_locked_leak"),
            ("pred05_rig","pred05_leak")
        ]:
            av=x[a].to_numpy().copy()
            bv=x[b].to_numpy().copy()
            aa=av.copy(); bb=bv.copy()
            aa[mask]=bv[mask]; bb[mask]=av[mask]
            x[a]=aa; x[b]=bb

        dd=macro_fold_metric(x,"leak",metric,locked)-macro_fold_metric(x,"rig",metric,locked)
        if np.isfinite(dd):
            valid+=1
            if abs(dd)>=abs(obs)-1e-15:
                extreme+=1

    return float((extreme+1)/(valid+1))

rows=[]
for arch,d in oof.groupby("architecture"):
    for metric in ALL_METRICS:
        locked=True
        rv=macro_fold_metric(d,"rig",metric,locked)
        lv=macro_fold_metric(d,"leak",metric,locked)
        off=sum(map(ord,arch+metric))
        lo,hi,bn=paired_cluster_bootstrap(d,metric,locked,seed=SEED+off)
        p=cluster_randomization_p(d,metric,locked,seed=SEED+123+off)
        rows.append({
            "Architecture":arch,"Metric":metric,
            "ThresholdPolicy":"validation-locked Youden J" if metric in THRESH_METRICS else "not applicable",
            "Rigorous":rv,"ControlledLeak":lv,"Delta_LeakMinusRig":lv-rv,
            "Delta_CI_Lo":lo,"Delta_CI_Hi":hi,
            "ClusterRandomization_p":p,"BootstrapReplicates":bn
        })

stats=pd.DataFrame(rows)
stats.to_csv(OUT/"primary_cluster_aware_metrics.csv",index=False)

# Supplementary fixed-0.5 thresholded metrics.
rows05=[]
for arch,d in oof.groupby("architecture"):
    for metric in THRESH_METRICS:
        rv=macro_fold_metric(d,"rig",metric,locked=False)
        lv=macro_fold_metric(d,"leak",metric,locked=False)
        rows05.append({
            "Architecture":arch,"Metric":metric,
            "Rigorous_Threshold0.5":rv,"ControlledLeak_Threshold0.5":lv,
            "Delta":lv-rv
        })
fixed05=pd.DataFrame(rows05)
fixed05.to_csv(OUT/"supplementary_fixed_0p5_threshold_metrics.csv",index=False)

print("\n=== FINAL PRIMARY RESULTS ===")
print(stats.to_string(index=False,float_format=lambda x:f"{x:.4f}"))
print("\nValidation-locked thresholds by fold:")
print(
    training_log[training_log.experiment=="primary"][
        ["architecture","fold","arm","locked_threshold","best_epoch"]
    ].to_string(index=False)
)


=== FINAL PRIMARY RESULTS ===
Architecture       Metric            ThresholdPolicy  Rigorous  ControlledLeak  Delta_LeakMinusRig  Delta_CI_Lo  Delta_CI_Hi  ClusterRandomization_p  BootstrapReplicates
ConvNeXtTiny          AUC             not applicable    0.8478          0.8488              0.0010      -0.0039       0.0060                  0.7821                 2000
ConvNeXtTiny       PR_AUC             not applicable    0.8279          0.8299              0.0020      -0.0063       0.0106                  0.7431                 2000
ConvNeXtTiny        Brier             not applicable    0.1624          0.1652              0.0028      -0.0016       0.0070                  0.2014                 2000
ConvNeXtTiny      LogLoss             not applicable    0.5072          0.5109              0.0037      -0.0091       0.0164                  0.5752                 2000
ConvNeXtTiny          ACC validation-locked Youden J    0.7767          0.7753             -0.0013      -0.0097       0

In [12]:
# =========================
# 12. Corrected Grad-CAM analysis
# =========================
def cluster_bootstrap_mean(df,value_col,B=BOOTSTRAPS,seed=SEED):
    pats=np.array(sorted(df.Patient_ID.astype(str).unique()))
    byp={p:df[df.Patient_ID.astype(str)==p] for p in pats}
    rng=np.random.default_rng(seed)
    vals=[]
    for _ in range(B):
        samp=rng.choice(pats,len(pats),replace=True)
        x=pd.concat([byp[p] for p in samp],ignore_index=True)
        vals.append(float(x[value_col].mean()))
    return (
        float(df[value_col].mean()),
        float(np.percentile(vals,2.5)),
        float(np.percentile(vals,97.5))
    )

def cluster_bootstrap_group_difference(df,value_col,group_col="exposed",B=BOOTSTRAPS,seed=SEED+500):
    pats=np.array(sorted(df.Patient_ID.astype(str).unique()))
    byp={p:df[df.Patient_ID.astype(str)==p] for p in pats}
    rng=np.random.default_rng(seed)
    vals=[]

    obs=(
        df.loc[df[group_col]==1,value_col].mean()
        -df.loc[df[group_col]==0,value_col].mean()
    )

    for _ in range(B):
        samp=rng.choice(pats,len(pats),replace=True)
        x=pd.concat([byp[p] for p in samp],ignore_index=True)
        if (x[group_col]==1).any() and (x[group_col]==0).any():
            vals.append(
                float(
                    x.loc[x[group_col]==1,value_col].mean()
                    -x.loc[x[group_col]==0,value_col].mean()
                )
            )
    return float(obs),float(np.percentile(vals,2.5)),float(np.percentile(vals,97.5))

xai_summary=[]

for metric in ["cam_cosine_similarity","cam_pearson_r","cam_centroid_shift"]:
    mean,lo,hi=cluster_bootstrap_mean(
        xai_pairs.dropna(subset=[metric]),metric,
        seed=SEED+900+sum(map(ord,metric))
    )
    diff,dlo,dhi=cluster_bootstrap_group_difference(
        xai_pairs.dropna(subset=[metric]),metric,
        seed=SEED+1200+sum(map(ord,metric))
    )
    xai_summary.append({
        "Metric":metric,
        "n_images":int(xai_pairs[metric].notna().sum()),
        "n_patients":int(xai_pairs.loc[xai_pairs[metric].notna(),"Patient_ID"].nunique()),
        "OverallMean":mean,"Overall_CI_Lo":lo,"Overall_CI_Hi":hi,
        "ExposedMinusUnexposed":diff,
        "ExposureDifference_CI_Lo":dlo,
        "ExposureDifference_CI_Hi":dhi,
    })

xai_stats=pd.DataFrame(xai_summary)
xai_stats.to_csv(XAI_DIR/"gradcam_cluster_summary.csv",index=False)

print("\n=== CORRECTED NIH GRAD-CAM SUMMARY ===")
print(xai_stats.to_string(index=False,float_format=lambda x:f"{x:.4f}"))
print(
    "\nInterpretation rule: these quantify attribution-map stability/change. "
    "They do NOT establish whether peripheral chest attention is clinically appropriate or inappropriate."
)


=== CORRECTED NIH GRAD-CAM SUMMARY ===
               Metric  n_images  n_patients  OverallMean  Overall_CI_Lo  Overall_CI_Hi  ExposedMinusUnexposed  ExposureDifference_CI_Lo  ExposureDifference_CI_Hi
cam_cosine_similarity       600         558       0.5913         0.5722         0.6092                -0.0069                   -0.0446                    0.0325
        cam_pearson_r       600         558       0.4738         0.4513         0.4963                -0.0103                   -0.0530                    0.0366
   cam_centroid_shift       600         558       0.1029         0.0973         0.1089                 0.0043                   -0.0075                    0.0169

Interpretation rule: these quantify attribution-map stability/change. They do NOT establish whether peripheral chest attention is clinically appropriate or inappropriate.


In [13]:
# =========================
# 13. Patch the native-prevalence-matched sensitivity pair
# =========================
native_stats=pd.DataFrame()
native_audit=pd.DataFrame()
native_training=[]

def safe_val_split(safe_df,fold_seed):
    n_splits=min(VAL_FOLDS,safe_df.Patient_ID.nunique())
    for off in range(50):
        sg=StratifiedGroupKFold(
            n_splits=n_splits,shuffle=True,random_state=fold_seed+off
        )
        tr_i,va_i=next(sg.split(safe_df,safe_df.Label,groups=safe_df.Patient_ID))
        tr=safe_df.iloc[tr_i].copy()
        va=safe_df.iloc[va_i].copy()
        if tr.Label.nunique()==2 and va.Label.nunique()==2:
            return tr,va
    raise RuntimeError("Could not reconstruct native sensitivity train/validation split.")

if RUN_NATIVE_PATCH:
    nat_path=PRIOR/"native_prevalence_matched_cohort_manifest.csv"
    if not nat_path.exists():
        raise FileNotFoundError(
            "The main run did not expose native_prevalence_matched_cohort_manifest.csv."
        )

    nat=enrich(pd.read_csv(nat_path))
    nat["row_id"]=nat.row_id.astype(int)

    # Reconstruct the same fixed 20% test assignment used by the main run.
    sk=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED+600)
    tr_idx,te_idx=next(sk.split(nat,nat.Label))
    ntest=nat.iloc[te_idx].copy().reset_index(drop=True)
    raw_train=nat.iloc[tr_idx].copy().reset_index(drop=True)

    test_ids=set(ntest.row_id.astype(int))
    test_pats=set(ntest.Patient_ID.astype(str))
    test_raw=set(ntest.raw_sha256.astype(str))
    test_dec=set(ntest.decoded_fp.astype(str))

    raw_train=raw_train[
        (~raw_train.row_id.astype(int).isin(test_ids))
        &(~raw_train.raw_sha256.astype(str).isin(test_raw))
        &(~raw_train.decoded_fp.astype(str).isin(test_dec))
    ].copy()

    safe=raw_train[
        ~raw_train.Patient_ID.astype(str).isin(test_pats)
    ].copy()

    nrig,nval=safe_val_split(safe,SEED+1000*101)
    nrig=nrig.reset_index(drop=True)
    nval=nval.reset_index(drop=True)

    nleak,ntest,naud=build_matched_leak_train(
        nat,nrig,nval,ntest,fold=101
    )
    native_audit=pd.DataFrame([naud])
    native_audit.to_csv(OUT/"native_phenotype_view_matched_audit.csv",index=False)

    # Identical class weights because phenotype/view matching preserves target counts.
    nw=class_weights(nrig)
    arm_pred={}

    for arm,train_df in [("rigorous",nrig),("controlled_leak",nleak)]:
        model,hist,best_epoch,threshold,val_pred=train_one(
            NATIVE_MODEL,train_df,nval,101,arm,
            class_weight=nw,experiment="native"
        )
        pp=predict_prob(model,make_loader(ntest,train=False,seed=SEED+101))
        pp["pred05"]=(pp.prob>=0.5).astype(int)
        pp["pred_locked"]=(pp.prob>=threshold).astype(int)
        pp["locked_threshold"]=threshold

        pp.to_csv(PRED_DIR/f"native_{arm}.csv",index=False)
        hist.to_csv(PRED_DIR/f"native_{arm}_history.csv",index=False)
        arm_pred[arm]=pp
        native_training.append({
            "experiment":"native","architecture":NATIVE_MODEL,"fold":101,"arm":arm,
            "best_epoch":best_epoch,"locked_threshold":threshold,
            "train_n":len(train_df),"val_n":len(nval),"test_n":len(ntest)
        })

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    r=arm_pred["rigorous"].rename(
        columns={
            "prob":"prob_rig","pred05":"pred05_rig","pred_locked":"pred_locked_rig",
            "locked_threshold":"threshold_rig"
        }
    )
    l=arm_pred["controlled_leak"][[
        "row_id","prob","pred05","pred_locked","locked_threshold"
    ]].rename(
        columns={
            "prob":"prob_leak","pred05":"pred05_leak","pred_locked":"pred_locked_leak",
            "locked_threshold":"threshold_leak"
        }
    )

    noof=ntest[[
        "row_id","Patient_ID","Label","Image Index","FindingProfile","View_Position",
        "n_train_siblings","n_same_profile_siblings"
    ]].merge(
        r[["row_id","prob_rig","pred05_rig","pred_locked_rig","threshold_rig"]],
        on="row_id",how="inner"
    ).merge(l,on="row_id",how="inner")
    noof["fold"]=1
    noof["architecture"]=NATIVE_MODEL
    noof.to_csv(OUT/"native_paired_predictions.csv",index=False)

    # Cluster-aware sensitivity metrics. Reuse primary statistics helpers.
    rows=[]
    for metric in ALL_METRICS:
        rv=macro_fold_metric(noof,"rig",metric,locked=True)
        lv=macro_fold_metric(noof,"leak",metric,locked=True)
        off=sum(map(ord,"native"+metric))
        lo,hi,bn=paired_cluster_bootstrap(noof,metric,locked=True,seed=SEED+1700+off)
        p=cluster_randomization_p(noof,metric,locked=True,seed=SEED+1800+off)
        rows.append({
            "Architecture":NATIVE_MODEL,"Metric":metric,
            "ThresholdPolicy":"validation-locked Youden J" if metric in THRESH_METRICS else "not applicable",
            "Rigorous":rv,"ControlledLeak":lv,"Delta_LeakMinusRig":lv-rv,
            "Delta_CI_Lo":lo,"Delta_CI_Hi":hi,
            "ClusterRandomization_p":p,
            "ObservedSamplePrevalence":float(nat.Label.mean())
        })
    native_stats=pd.DataFrame(rows)
    native_stats.to_csv(OUT/"native_cluster_aware_metrics.csv",index=False)

    print("\n=== PATCHED NATIVE-PREVALENCE SENSITIVITY ===")
    print(native_audit.to_string(index=False))
    print(native_stats.to_string(index=False,float_format=lambda x:f"{x:.4f}"))
else:
    print("Native patch disabled.")

native | ResNet50 | fold 101 | rigorous | train=3223 val=821
  epoch 01/12 loss=0.6218 valAUC=0.8279 time=141s
  epoch 02/12 loss=0.5187 valAUC=0.7987 time=87s
  epoch 03/12 loss=0.4626 valAUC=0.8386 time=86s
  epoch 04/12 loss=0.4287 valAUC=0.8441 time=87s
  epoch 05/12 loss=0.3920 valAUC=0.8277 time=86s
  epoch 06/12 loss=0.3371 valAUC=0.8493 time=86s
  epoch 07/12 loss=0.2883 valAUC=0.8313 time=86s
  epoch 08/12 loss=0.2190 valAUC=0.8128 time=86s
  epoch 09/12 loss=0.1878 valAUC=0.8241 time=85s
  early stop
native | ResNet50 | fold 101 | controlled_leak | train=3223 val=821
  epoch 01/12 loss=0.6242 valAUC=0.8324 time=96s
  epoch 02/12 loss=0.5220 valAUC=0.8498 time=86s
  epoch 03/12 loss=0.4758 valAUC=0.8484 time=87s
  epoch 04/12 loss=0.4183 valAUC=0.8469 time=86s
  epoch 05/12 loss=0.3523 valAUC=0.8301 time=87s
  early stop

=== PATCHED NATIVE-PREVALENCE SENSITIVITY ===
 fold  test_n  test_patients  rig_train_n  leak_train_n  val_n  eligible_sibling_candidates  used_sibling_candi

In [14]:
# =========================
# 14. Final integrity summary + manuscript tables + archive
# =========================
if native_training:
    training_log=pd.concat(
        [training_log,pd.DataFrame(native_training)],
        ignore_index=True
    )
    training_log.to_csv(OUT/"training_summary.csv",index=False)

summary={
    "source_main_run_output":str(PRIOR),
    "raw_data_root":str(ROOT),
    "old_predictions_reused":False,
    "old_model_checkpoints_reused":False,
    "old_xai_reused":False,
    "primary_cohort_images":int(len(cohort)),
    "primary_cohort_patients":int(cohort.Patient_ID.nunique()),
    "folds":FOLDS,
    "architectures":RUN_ARCHITECTURES,
    "exact_full_finding_profile_plus_view_matched":bool(
        audit_df.exact_match_strata_preserved.all()
    ),
    "mean_controlled_test_exposure_pct":float(100*audit_df.test_exposure_pct.mean()),
    "mean_candidate_utilization_pct":float(audit_df.candidate_utilization_pct.mean()),
    "all_rigorous_train_test_patient_overlap_zero":bool(
        audit_df.rig_train_test_patient_overlap.eq(0).all()
    ),
    "all_test_rawhash_overlap_zero":bool(
        (
            audit_df.rig_train_test_rawhash_overlap.eq(0)
            &audit_df.leak_train_test_rawhash_overlap.eq(0)
        ).all()
    ),
    "all_test_decoded_overlap_zero":bool(
        (
            audit_df.rig_train_test_decoded_overlap.eq(0)
            &audit_df.leak_train_test_decoded_overlap.eq(0)
        ).all()
    ),
    "deterministic_runtime":versions,
    "threshold_policy":"Youden J selected on each model's validation predictions and locked before test evaluation",
    "xai_architecture":XAI_ARCHITECTURE,
    "xai_method":"paired true-class Grad-CAM map similarity / centroid displacement",
    "xai_quantitative_images":int(len(xai_pairs)),
    "xai_visual_case_selection":"seeded label/view-stratified selection before predictions",
    "native_patch_run":bool(RUN_NATIVE_PATCH),
}

with open(OUT/"final_integrity_summary.json","w") as f:
    json.dump(summary,f,indent=2)

assert summary["exact_full_finding_profile_plus_view_matched"]
assert summary["all_rigorous_train_test_patient_overlap_zero"]
assert summary["all_test_rawhash_overlap_zero"]
assert summary["all_test_decoded_overlap_zero"]
assert summary["old_predictions_reused"] is False
assert summary["old_model_checkpoints_reused"] is False

primary=stats[
    stats.Metric.isin(["AUC","PR_AUC","ACC","Macro_F1","Brier"])
].copy()
primary.to_csv(OUT/"manuscript_primary_result_table.csv",index=False)

print("\n=== FINAL INTEGRITY SUMMARY ===")
print(json.dumps(summary,indent=2))
print("\n=== MANUSCRIPT-READY PRIMARY TABLE ===")
print(primary.to_string(index=False,float_format=lambda x:f"{x:.4f}"))

readme=f"""NIH FINAL PATCH — PHENOTYPE/VIEW MATCHING + DETERMINISM + VALIDATION-LOCKED THRESHOLDS + CORRECTED XAI

Inputs
- prior executed main-run output: {PRIOR}
- raw NIH image root: {ROOT}

Final primary design
- frozen 6,000-image cohort and fixed rigorous/validation/test manifests reused;
- old predictions/checkpoints/XAI not reused;
- controlled-leak replacements matched exactly on the entire normalized Finding Labels profile + View Position;
- candidate rows without an exact safe profile/view match are excluded rather than creating a composition confound;
- demographics are used as a secondary greedy matching preference and audited;
- strict deterministic paired rerun of ResNet-50 and ConvNeXt-Tiny;
- operating thresholds selected on validation only via Youden J, then locked before test evaluation;
- fixed 0.5 metrics exported separately as supplementary results;
- patient-cluster bootstrap CIs and paired randomization tests;
- corrected ResNet-50 Grad-CAM on a preselected OOF subset;
- XAI quantifies map similarity/centroid shift, not 'border artifact' pathology claims;
- native-prevalence-matched sensitivity pair patched with the same final control design.

Key files
- phenotype_view_matched_leakage_audit.csv
- primary_oof_paired_predictions.csv
- primary_cluster_aware_metrics.csv
- supplementary_fixed_0p5_threshold_metrics.csv
- manuscript_primary_result_table.csv
- native_phenotype_view_matched_audit.csv
- native_paired_predictions.csv
- native_cluster_aware_metrics.csv
- training_summary.csv
- final_integrity_summary.json
- xai/preselected_quantitative_xai_cases.csv
- xai/preselected_visual_cases.csv
- xai/gradcam_paired_map_metrics.csv
- xai/gradcam_cluster_summary.csv
- xai/*overlay PNGs
"""

(OUT/"README.txt").write_text(readme)

archive=BASE/"NIH_FinalPatch_Results.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p,p.relative_to(BASE))

print("\nFinal archive:",archive)
print("NIH FINAL PATCH COMPLETE.")


=== FINAL INTEGRITY SUMMARY ===
{
  "source_main_run_output": "/kaggle/working/_prior_nih_results/nih_corrected_leakage_audit",
  "raw_data_root": "/kaggle/input",
  "old_predictions_reused": false,
  "old_model_checkpoints_reused": false,
  "old_xai_reused": false,
  "primary_cohort_images": 6000,
  "primary_cohort_patients": 3911,
  "folds": 5,
  "architectures": [
    "ResNet50",
    "ConvNeXtTiny"
  ],
  "exact_full_finding_profile_plus_view_matched": true,
  "mean_controlled_test_exposure_pct": 46.150000000000006,
  "mean_candidate_utilization_pct": 89.70175225072651,
  "all_rigorous_train_test_patient_overlap_zero": true,
  "all_test_rawhash_overlap_zero": true,
  "all_test_decoded_overlap_zero": true,
  "deterministic_runtime": {
    "python": "3.13.15",
    "torch": "2.11.0+cu128",
    "torchvision": "0.26.0+cu128",
    "cuda": "12.8",
    "device": "Tesla T4",
    "cudnn_deterministic": true,
    "cudnn_benchmark": false,
    "deterministic_algorithms": true,
    "CUBLAS_WORK